# User Experience para ciência de dados
Previsão de atrasos em pedidos.

In [68]:
import pandas as pd

# Força limpeza
import importlib
import data_preparation_final
importlib.reload(data_preparation_final)

# Importar lógica de preparação de dados existente
from data_preparation_final import load_and_clean_data

try:
    from ydata_profiling import ProfileReport
    _profiling_available = True
except ImportError:
    _profiling_available = False
    print("ydata_profiling indisponível (conflito de dependência). O profiling será ignorado.")

In [69]:
def prepare_data_for_lgbm(df):
    """
    Limpeza e filtragem sugerida para o modelo LightGBM.
    """
    print("\n--- Preparação de Dados para LightGBM ---")
    inicial = len(df)
    
    # 1. Limpeza Crítica: dropna em campos fundamentais
    cols_limpeza = ['dt_despacho_pedido', 'dt_entrega_pedido', 'dt_pagamento_pedido', 'qtd_dias_tat']
    
    # --- OPÇÃO DE IMPUTAÇÃO (COMENTADA) ---
    # Se decidirmos não dropar os nulos de dt_pagamento_pedido (6.4%):
    # df['dt_pagamento_pedido'] = df['dt_pagamento_pedido'].fillna(method='ffill') # ou uma data fixa 'Pendente'
    # df['dias_aprovacao'] = df['dias_aprovacao'].fillna(-1) # Categoria para 'Pendente'
    # --------------------------------------
    
    df = df.dropna(subset=cols_limpeza).copy()
    
    posterior = len(df)
    print(f"Registros antes: {inicial}")
    print(f"Registros após limpeza (dropna): {posterior}")
    print(f"Perda de dados: {1 - (posterior/inicial):.2%}")

    # 2. Tratamento de Outliers (Percentil 99 em qtd_dias_tat)
    limite_99 = df['qtd_dias_tat'].quantile(0.99)
    df = df[df['qtd_dias_tat'] <= limite_99].copy()
    print(f"Outliers removidos (TAT > {limite_99:.1f} dias): {posterior - len(df)}")

    return df

## Limpeza e tratamento de dados

In [70]:
# 1. Carregar e Limpar
input_file = "pedidos_logistica.parquet"
df = load_and_clean_data(input_file, drop_ids=False)

if df is not None:
    df = prepare_data_for_lgbm(df)

Carregando dados de pedidos_logistica.parquet...
Colunas após renomeação:
id, cod_pedido, uf
grp_transportadora, dt_despacho_pedido, dt_entrega_pedido
dt_previsao_entrega_cliente, dt_criacao, dt_pagamento_pedido
flg_existem_ocorrencias, tp_praca, des_unidade_negocio
des_cd_origem, qtd_dias_tat, tp_performance_entrega
cidade_destinatario
Removendo registros inconsistentes (entrega antecede despacho): 7
Realizando engenharia de features...
Processamento concluído. Formato final: (490210, 21)
Salvando dataset limpo em pedidos_logistica_limpo.parquet...

--- Preparação de Dados para LightGBM ---
Registros antes: 490210
Registros após limpeza (dropna): 458687
Perda de dados: 6.43%
Outliers removidos (TAT > 14.0 dias): 3596


In [71]:
# Manter para ter coerencia com 'trabalho' - Pedro
df = df.drop(
    columns=[
        # 'row_id',
        # 'hr_despacho_pedido',
        'dt_entrega_pedido',
        # 'hr_entrega_pedido',
        'flg_existem_ocorrencias'
    ],
    errors='ignore'
 )

df.head(10)

,id,cod_pedido,uf,grp_transportadora,dt_despacho_pedido,dt_previsao_entrega_cliente,dt_criacao,dt_pagamento_pedido,tp_praca,des_unidade_negocio,des_cd_origem,qtd_dias_tat,tp_performance_entrega,cidade_destinatario,dias_gastos_cd,dias_transito,dias_atraso_real,dias_ciclo,dias_restantes_prazo
0,67044,127510252-1,RJ,Transportadora 3,2023-11-27 14:57:25,2023-12-06,2023-11-24,2023-11-24,Capital,Multi,PR-Campina G. Sul,5.0,1,RIO DE JANEIRO,3.623206,4.209201,-4.167593,7.832407,12.0
1,67045,122170353-1,SE,Transportadora 3,2023-07-19 11:41:44,2023-07-27,2023-07-17,2023-07-17,Capital,Mono,PR-Campina G. Sul,6.0,1,ARACAJU,2.487315,6.127512,-1.385174,8.614826,10.0
2,67046,125676313-1,RN,Transportadora 2,2023-11-08 15:44:19,2023-11-20,2023-11-07,2023-11-07,Interior,Mono,PR-Campina G. Sul,7.0,1,NATAL,1.655775,9.878773,-1.465451,11.534549,13.0
3,67047,124809810-1,PA,Transportadora 2,2023-10-16 12:40:54,2023-10-25,2023-10-14,2023-10-14,Reg. Metropolitana,Multi,PR-Campina G. Sul,6.0,1,ANANINDEUA,2.528403,6.918843,-1.552755,9.447245,11.0
5,67049,124134539,MG,Transportadora 1,2023-09-25 08:47:42,2023-10-02,2023-09-24,2023-09-24,Interior,Multi,PR-Campina G. Sul,4.0,1,JUIZ DE FORA,1.366458,3.065231,-3.568310,4.431690,8.0
6,67050,123148738-1,MG,Transportadora 1,2023-08-22 14:13:44,2023-08-30,2023-08-21,2023-08-21,Interior,Multi,PR-Campina G. Sul,5.0,1,BAEPENDI,1.592870,6.010185,-1.396944,7.603056,9.0
7,67051,127085176-1,TO,Transportadora 2,2023-11-24 22:14:31,2023-12-06,2023-11-22,2023-11-22,Capital,Mono,SP-Registro,6.0,1,PALMAS,2.926748,5.557639,-5.515613,8.484387,14.0
8,67052,122966837-3,RJ,Transportadora 1,2023-08-16 14:53:27,2023-08-21,2023-08-15,2023-08-15,Reg. Metropolitana,Multi,PR-Campina G. Sul,3.0,1,NITEROI,1.620451,2.031829,-2.347720,3.652280,6.0
9,67053,122721965-1,MG,Transportadora 1,2023-08-07 09:41:30,2023-08-11,2023-08-07,2023-08-06,Interior,Multi,PR-Campina G. Sul,3.0,1,POCOS DE CALDAS,1.403819,2.162836,-1.433345,2.566655,5.0
10,67054,126966272,AC,Transportadora 1,2023-11-23 01:13:53,2023-12-11,2023-11-21,2023-11-21,Capital,Multi,PR-Campina G. Sul,9.0,1,RIO BRANCO,2.051308,11.681597,-6.267095,13.732905,20.0


## Feature Engeeniring

In [72]:
import numpy as np

# --- 1. Temporal features from dispatch date ---
df['dia_semana_despacho'] = df['dt_despacho_pedido'].dt.dayofweek
df['mes_despacho']        = df['dt_despacho_pedido'].dt.month
df['hora_despacho'] = df['dt_despacho_pedido'].dt.hour
# float32 handles NaT-derived NaN natively (unlike int32)
df['semana_ano']          = df['dt_despacho_pedido'].dt.isocalendar().week.astype('float32')

# Weekend dispatch flag (Sat=5, Sun=6): carrier networks are thinner on weekends
df['is_fds_despacho']   = (df['dia_semana_despacho'] >= 5).astype('int8')

# High-demand season: November (Black Friday) and December (Christmas)
df['is_alta_temporada'] = df['mes_despacho'].isin([11, 12]).astype('int8')

# Dispatch shift — earlier dispatches tend to have better same-day processing
if 'hora_despacho' in df.columns:
    df['turno_despacho'] = pd.cut(
        df['hora_despacho'],
        bins=[-1, 5, 11, 17, 23],
        labels=['Madrugada', 'Manha', 'Tarde', 'Noite']
    ).astype('category')

# --- 2. Derived date-ratio features ---
df['dias_criacao_pagamento'] = (df['dt_pagamento_pedido'] - df['dt_criacao']).dt.days

# Days the carrier has from dispatch to expected delivery
df['prazo_apos_despacho'] = (
    df['dt_previsao_entrega_cliente'] - df['dt_despacho_pedido']
).dt.days

# Fraction of total deadline already consumed inside the CD (>1 = deadline already blown)
df['ratio_cd_prazo'] = (
    df['dias_gastos_cd'] / df['dias_restantes_prazo'].replace(0, np.nan)
).clip(0, 2)

# Net delivery margin: negative = deadline blown before dispatch
df['margem_entrega'] = df['prazo_apos_despacho'] - df['dias_gastos_cd']

# --- Summary ---
new_cols = [
    'dia_semana_despacho', 'mes_despacho', 'semana_ano',
    'is_fds_despacho', 'is_alta_temporada',
    'dias_criacao_pagamento', 'prazo_apos_despacho',
    'ratio_cd_prazo', 'margem_entrega',
]
if 'hora_despacho' in df.columns:
    new_cols = ['hora_despacho', 'turno_despacho'] + new_cols

print(f'Novas features adicionadas ({len(new_cols)}):')
print(df[new_cols].describe(include='number').T[['count', 'mean', 'min', 'max']].to_string())
print("\n")
print(df.describe(include=['category']))

Novas features adicionadas (11):
                           count       mean        min       max
hora_despacho           455091.0  11.364288   0.000000  23.00000
dia_semana_despacho     455091.0   2.391864   0.000000   6.00000
mes_despacho            455091.0   9.568350   1.000000  12.00000
semana_ano              455091.0  39.868713   1.000000  52.00000
is_fds_despacho         455091.0   0.136177   0.000000   1.00000
is_alta_temporada       455091.0   0.428723   0.000000   1.00000
dias_criacao_pagamento  455091.0  -0.064363 -19.000000  15.00000
prazo_apos_despacho     455091.0   6.340029 -16.000000  73.00000
ratio_cd_prazo          455075.0   0.296310   0.000000   2.00000
margem_entrega          455091.0   4.180788 -35.697199  71.44875


            uf grp_transportadora  tp_praca des_unidade_negocio  \
count   455091             455091    455091              455091   
unique      27                  8         3                   2   
top         SP   Transportadora 1  Interior      

## Profiling de dados

In [79]:
# if _profiling_available:
#     profile = ProfileReport(df, title="Profiling Report")
#     html = profile.to_html()
#     output_file = 'report.html'
#     with open(output_file, 'w') as f:
#         f.write(html)
#     print(f"Relatório salvo em {output_file}")
# else:
#     print("Profiling ignorado — instale uma versão compatível de numba/numpy para habilitar.")

In [93]:

import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ---------------------------------------------------------
# Step 3: Split Features (X) and Target (y)
# ---------------------------------------------------------
# selected_features = [
#     'cidade_destinatario',
#     'uf',
#     'grp_transportadora',
#     'dt_previsao_entrega_cliente',
#     'dt_criacao',
#     'dt_pagamento_pedido',
#     'tp_praca',
#     'des_unidade_negocio',
#     'des_cd_origem',
# ]
selected_features = [
    # Original features
    'cidade_destinatario',
    'uf',
    'grp_transportadora',
    'dt_previsao_entrega_cliente',
    'dt_criacao',
    'dt_pagamento_pedido',
    'tp_praca',
    'des_unidade_negocio',
    'des_cd_origem',
    'dias_gastos_cd',
    'dias_restantes_prazo',
    # Feature engineering — temporal
    # 'hora_despacho',
    'turno_despacho',
    'dia_semana_despacho',
    'mes_despacho',
    'semana_ano',
    'is_fds_despacho',
    'is_alta_temporada',
    # Feature engineering — ratios
    'dias_criacao_pagamento',
    'prazo_apos_despacho',
    'ratio_cd_prazo',
    'margem_entrega',
]

# 1. Divisão Temporal 70/30
df = df.sort_values('dt_criacao')
split_idx = int(len(df) * 0.7)
# df de treinamento e avaliação do modelo
train_test_data = df.iloc[:split_idx].copy()
# df para simulação (dados não conhecidos pelo modelo)
holdout_data = df.iloc[split_idx:].copy()

available_features = [col for col in selected_features if col in train_test_data.columns]
missing_features = [col for col in selected_features if col not in train_test_data.columns]

if missing_features:
    print('Missing columns (ignored):', missing_features)

X = train_test_data[available_features].copy()
y = train_test_data['tp_performance_entrega']

# Remove rows with missing target
valid_mask = y.notna()
X = X.loc[valid_mask].copy()
y = y.loc[valid_mask].astype('int32').copy()

# Convert requested date columns to numeric representation (ordinal days)
# date_cols = ['dt_previsao_entrega_cliente', 'dt_criacao', 'dt_pagamento_pedido']
# for col in [c for c in date_cols if c in X.columns]:
#     X[col] = pd.to_datetime(X[col], errors='coerce')
#     X[col] = X[col].map(lambda x: x.toordinal() if pd.notna(x) else np.nan).astype('float32')

# !!! Tratamento de Datas (Ponto de Atenção)
# Você está convertendo datas para toordinal(). Para logística, o número ordinal puro (ex: 738900) 
# é difícil para o modelo entender sazonalidade. O ganho de informação (gain) nessas colunas costuma ser baixo, 
# o que contribui para o erro de "no further splits".
# Dica para o Semáforo: Em vez de apenas o número ordinal, crie colunas de diferença (lead time):
# Exemplo de Feature Engineering útil para logística:
# X['dias_previsao_criacao'] = (X['dt_previsao_entrega_cliente'] - X['dt_criacao'])
# X['dias_pagamento_criacao'] = (X['dt_pagamento_pedido'] - X['dt_criacao'])

# Encode requested categorical columns as numeric codes
categorical_cols = [
    'cidade_destinatario',
    'uf',
    'grp_transportadora',
    'tp_praca',
    'des_unidade_negocio',
    'des_cd_origem',
    # Novas categorias
    'turno_despacho',
]
# for col in [c for c in categorical_cols if c in X.columns]:
#     X[col] = X[col].astype('category').cat.codes.replace(-1, np.nan).astype('float32')
# !!! Categorização Manual vs. Nativa
# Você está usando .cat.codes. O LightGBM tem um suporte nativo excelente para categorias que performa muito melhor do que converter para números ordinais.
# Como melhorar:
# Mantenha as colunas como o tipo category do pandas.
# Não use .cat.codes.
# Passe a lista de nomes das colunas para o modelo.
# No seu loop de categorias, pare aqui:
for col in categorical_cols:
    X[col] = X[col].astype('category')

# LightGBM does not accept object/datetime columns directly
unsupported_cols = X.select_dtypes(include=['object', 'datetime64[ns]', 'datetimetz']).columns
if len(unsupported_cols) > 0:
    print('Dropping unsupported columns:', list(unsupported_cols))
X = X.drop(columns=unsupported_cols, errors='ignore')

print('Training columns:', list(X.columns))

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------------
# Step 4: Initialize and Train the Model
# ---------------------------------------------------------
# model = lgb.LGBMClassifier(
#     n_estimators=100,
#     learning_rate=0.1,
#     max_depth=5,
#     random_state=42,
#     is_unbalance=True
# )
# Sobre a causa das mensagens " LightGBM Warning No further splits with positive gain, best gain: -inf"
# Com mais de 400 mil registros, o problema certamente não é falta de dados (o min_child_samples=20 padrão seria atendido facilmente). 
# O aviso aparece porque você está limitando muito o modelo com max_depth=5 em um cenário de logística que parece ser complexo.
# Com mais de 400k linhas, uma profundidade de 5 níveis (máximo de 32 folhas) é muito pouco para capturar as nuances de logística 
# (como variações por cidade ou transportadora). O LightGBM tenta criar divisões, mas como ele já atingiu o limite de profundidade ou 
# as combinações restantes não batem com o is_unbalance=True, ele desiste e gera o aviso.

# Sugestão: Deixe o modelo crescer mais e controle pelo número de folhas, que é mais eficiente no LightGBM.
model = lgb.LGBMClassifier(
    is_unbalance=True,
    n_estimators=200,      # Aumente um pouco já que tem muitos dados
    learning_rate=0.05,    # Reduza a taxa para aprender com mais calma
    num_leaves=63,         # Aumente a complexidade (2^max_depth - 1)
    max_depth=-1,          # Deixe o crescimento livre (controlado por num_leaves)
    random_state=42,    
    verbose=-1             # Silencie o aviso agora que ajustamos a estrutura
)

# Fit model
model.fit(X_train, y_train, categorical_feature=categorical_cols)

# ---------------------------------------------------------
# Step 5: Make Predictions (The "Risk Score")
# ---------------------------------------------------------
predictions_binary = model.predict(X_test)
predictions_proba = model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# Step 6: Evaluate Model Quality
# ---------------------------------------------------------
accuracy = accuracy_score(y_test, predictions_binary)
precision = precision_score(y_test, predictions_binary, zero_division=0)
recall = recall_score(y_test, predictions_binary, zero_division=0)
f1 = f1_score(y_test, predictions_binary, zero_division=0)
roc_auc = roc_auc_score(y_test, predictions_proba)
cm = confusion_matrix(y_test, predictions_binary)

print('=== Model Metrics ===')
print(f'Accuracy : {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1-score : {f1:.4f}')
print(f'ROC-AUC  : {roc_auc:.4f}')
print('\nConfusion Matrix:')
print(cm)
print('\nClassification Report:')
print(classification_report(y_test, predictions_binary, zero_division=0))

Dropping unsupported columns: ['dt_previsao_entrega_cliente', 'dt_criacao', 'dt_pagamento_pedido']
Training columns: ['cidade_destinatario', 'uf', 'grp_transportadora', 'tp_praca', 'des_unidade_negocio', 'des_cd_origem', 'dias_gastos_cd', 'dias_restantes_prazo', 'turno_despacho', 'dia_semana_despacho', 'mes_despacho', 'semana_ano', 'is_fds_despacho', 'is_alta_temporada', 'dias_criacao_pagamento', 'prazo_apos_despacho', 'ratio_cd_prazo', 'margem_entrega']
=== Model Metrics ===
Accuracy : 0.8777
Precision: 0.9864
Recall   : 0.8857
F1-score : 0.9333
ROC-AUC  : 0.8460

Confusion Matrix:
[[ 1369   754]
 [ 7038 54552]]

Classification Report:
              precision    recall  f1-score   support

           0       0.16      0.64      0.26      2123
           1       0.99      0.89      0.93     61590

    accuracy                           0.88     63713
   macro avg       0.57      0.77      0.60     63713
weighted avg       0.96      0.88      0.91     63713



In [98]:
# import matplotlib.pyplot as plt

# fig, ax = plt.subplots(figsize=(10, 6))
# lgb.plot_importance(model, ax=ax)
# plt.tight_layout()
# plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
# plt.close(fig)
# print('Saved: feature_importance.png')

In [99]:
# ---------------------------------------------------------
# Step 7: View the Results
# ---------------------------------------------------------
results_df = X_test.copy()

# predict_proba[:, 1] = probabilidade de entrega NO PRAZO (classe 1)
# O risco de atraso é o complemento: 1 - probabilidade_no_prazo
results_df['probabilidade_no_prazo'] = predictions_proba
results_df['probabilidade_atraso']   = 1.0 - predictions_proba

# Semáforo baseado no RISCO DE ATRASO (não na probabilidade de entrega no prazo)
# Verde   → risco < 30%  (entrega confortável)
# Amarelo → risco 30–70% (atenção)
# Vermelho → risco > 70% (crítico)
results_df['risco_semaforo'] = pd.cut(
    results_df['probabilidade_atraso'],
    bins=[-0.1, 0.3, 0.7, 1.1],
    labels=['🟢 Verde', '🟡 Amarelo', '🔴 Vermelho']
)
print(results_df[['probabilidade_no_prazo', 'probabilidade_atraso', 'risco_semaforo']].head(20))

In [100]:
import joblib

# Build categorical mappings from training data
categorical_mappings = {}
for col in [c for c in categorical_cols if c in train_test_data.columns]:
    cats = pd.Series(train_test_data.loc[valid_mask, col].astype("string").dropna().unique()).sort_values().tolist()
    categorical_mappings[col] = {v: i for i, v in enumerate(cats)}

bundle = {
    "model": model,
    "selected_features": selected_features,
    "date_cols": ["dt_previsao_entrega_cliente", "dt_criacao", "dt_pagamento_pedido"],
    "categorical_cols": [c for c in categorical_cols if c in selected_features],
    "categorical_mappings": categorical_mappings,
    "threshold": 0.5
}

joblib.dump(bundle, "model_bundle.joblib")
print("Saved model_bundle.joblib")

Saved model_bundle.joblib


In [101]:
# df[df['tp_performance_entrega'] == 0]

# Da parte do df(70%) utilizado no treino/teste
train_test_data[train_test_data['tp_performance_entrega'] == 0]
# len(train_test_data)

,id,cod_pedido,uf,grp_transportadora,dt_despacho_pedido,dt_previsao_entrega_cliente,dt_criacao,dt_pagamento_pedido,tp_praca,des_unidade_negocio,...,mes_despacho,hora_despacho,semana_ano,is_fds_despacho,is_alta_temporada,turno_despacho,dias_criacao_pagamento,prazo_apos_despacho,ratio_cd_prazo,margem_entrega
420792,489065,121525499-1,GO,Transportadora 4,2023-07-01 10:20:27,2023-06-30,2023-07-01,2023-06-25,Capital,Multi,...,7,10,26.0,1,0,Manha,-6,-2,1.286174,-8.430868
420241,488506,121666681-1,BA,Transportadora 2,2023-07-03 20:33:48,2023-07-10,2023-07-01,2023-07-01,Capital,Mono,...,7,20,27.0,0,0,Noite,0,6,0.317423,3.143194
428520,496872,121669260-1,MA,Transportadora 2,2023-07-03 20:47:07,2023-07-13,2023-07-01,2023-07-01,Capital,Mono,...,7,20,27.0,0,0,Noite,0,9,0.238838,6.133947
415771,483999,121674017-1,RJ,Transportadora 1,2023-07-03 14:01:46,2023-07-06,2023-07-01,2023-07-01,Capital,Mono,...,7,14,27.0,0,0,Tarde,0,2,0.516912,-0.584560
420398,488663,121671849-1,PB,Transportadora 3,2023-07-03 16:29:56,2023-07-10,2023-07-01,2023-07-02,Capital,Mono,...,7,16,27.0,0,0,Tarde,1,6,0.210932,4.312546
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
417891,486131,126206866-1,GO,Transportadora 2,2023-11-18 15:30:19,2023-11-27,2023-11-14,2023-11-14,Interior,Multi,...,11,15,46.0,1,1,Tarde,0,8,0.357389,3.353947
426375,494707,126187628,SP,Transportadora 8,2023-11-16 18:40:30,2023-11-17,2023-11-14,2023-11-14,Capital,Multi,...,11,18,46.0,0,1,Noite,0,0,0.926042,-2.778125
433500,501902,126248826,RJ,Transportadora 3,2023-11-15 23:56:30,2023-11-22,2023-11-14,2023-11-14,Capital,Multi,...,11,23,46.0,0,1,Noite,0,6,0.249696,4.002431
426115,494445,126257961-1,SP,Transportadora 1,2023-11-17 19:21:07,2023-11-21,2023-11-14,2023-11-14,Capital,Multi,...,11,19,46.0,0,1,Noite,0,3,0.543762,-0.806331
